# Indexing Arrays

Blosc2 can attach indexes to 1-D `NDArray` objects and to fields inside 1-D structured arrays. These indexes accelerate selective queries, and `full` indexes can also drive ordered access directly through `sort(order=...)`, `indices(order=...)`, and `itersorted(...)`.

This tutorial covers:

- how to create field and expression indexes,
- how to tell whether a query is using an index,
- what sort of acceleration different index kinds can deliver on a selective query,
- how index persistence works,
- when to rebuild indexes,
- and a recommended workflow for keeping append-heavy `full` indexes compact.


## Setup

In [1]:
import statistics
import time
from pathlib import Path

import numpy as np

import blosc2


def show_index_summary(label, descriptor):
    print(
        f"{label}: kind={descriptor['kind']}, persistent={descriptor['persistent']}, "
        f"ooc={descriptor['ooc']}, stale={descriptor['stale']}"
    )


def explain_subset(expr):
    info = expr.explain()
    keep = {}
    for key in ("will_use_index", "reason", "kind", "level", "lookup_path", "full_runs"):
        if key in info:
            keep[key] = info[key]
    return keep


def median_ms(func, repeats=5, warmup=1):
    for _ in range(warmup):
        func()
    samples = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        func()
        samples.append((time.perf_counter() - t0) * 1e3)
    return statistics.median(samples)


paths = [
    Path("indexing_tutorial_medium.b2nd"),
    Path("indexing_tutorial_append_full.b2nd"),
]
for path in paths:
    blosc2.remove_urlpath(path)

## Index kinds and how to create them

Blosc2 currently supports four index kinds:

- `ultralight`: compact summaries only,
- `light`: summary levels plus lightweight per-block payloads,
- `medium`: richer exact-filter payloads,
- `full`: globally sorted payloads for exact filtering and ordered reuse.

There is one active index per target field or expression. If you create another index on the same target, it replaces the previous one. The easiest way to compare kinds is to build them on separate arrays.

This example uses one million rows so the timing differences are visible without turning the tutorial into a long benchmark.

In [5]:
N_ROWS = 10_000_000
QUERY_TEXT = "(id >= -25.0) & (id < 25.0)"

rng = np.random.default_rng(0)
dtype = np.dtype([("id", np.float64), ("payload", np.int32)])
data = np.zeros(N_ROWS, dtype=dtype)
# Build a predictable id column, then shuffle it so the source data is not already ordered.
data["id"] = np.arange(data.shape[0], dtype=np.float64) - data.shape[0] / 2
rng.shuffle(data["id"])
data["payload"] = np.arange(data.shape[0], dtype=np.int32)

chunks = (250_000,)
blocks = (50_000,)

indexed_arrays = {}
for kind in ("ultralight", "light", "medium", "full"):
    arr = blosc2.asarray(data.copy(), chunks=chunks, blocks=blocks)
    descriptor = arr.create_index(field="id", kind=kind)
    indexed_arrays[kind] = arr
    show_index_summary(kind, descriptor)

ultralight: kind=ultralight, persistent=False, ooc=False, stale=False
light: kind=light, persistent=False, ooc=True, stale=False
medium: kind=medium, persistent=False, ooc=True, stale=False
full: kind=full, persistent=False, ooc=True, stale=False


## Using an index for filtering

Range predicates are planned automatically when you use `where(...)`. You can inspect the plan with `explain()` and compare the indexed result with a scan by passing `_use_index=False` to `compute()`.

In [6]:
medium_arr = indexed_arrays["medium"]
expr = blosc2.lazyexpr(QUERY_TEXT, medium_arr.fields).where(medium_arr)

print(explain_subset(expr))

indexed = expr.compute()[:]
scanned = expr.compute(_use_index=False)[:]
np.testing.assert_array_equal(indexed, scanned)
print(f"Matched rows: {len(indexed)}")

{'will_use_index': True, 'reason': 'multi-field exact indexes selected', 'kind': 'medium', 'level': 'exact', 'lookup_path': None, 'full_runs': 0}
Matched rows: 50


### Timing the query with and without indexes

The next cell measures the same selective predicate on all four index kinds and compares it with a forced full scan. On this exact workload, `medium` and `full` usually show the clearest benefit because they carry richer exact-filter payloads.

In [7]:
timing_rows = []
expected = None
for kind, arr in indexed_arrays.items():
    expr = blosc2.lazyexpr(QUERY_TEXT, arr.fields).where(arr)
    result = expr.compute()[:]
    if expected is None:
        expected = result
    else:
        np.testing.assert_array_equal(result, expected)

    scan_ms = median_ms(lambda expr=expr: expr.compute(_use_index=False)[:], repeats=3)
    index_ms = median_ms(lambda expr=expr: expr.compute()[:], repeats=3)
    timing_rows.append((kind, scan_ms, index_ms, scan_ms / index_ms))

print(f"Selective filter over {N_ROWS:,} rows")
print(f"{'kind':<12} {'scan_ms':>10} {'index_ms':>10} {'speedup':>10}")
for kind, scan_ms, index_ms, speedup in timing_rows:
    print(f"{kind:<12} {scan_ms:10.3f} {index_ms:10.3f} {speedup:10.2f}x")

Selective filter over 10,000,000 rows
kind            scan_ms   index_ms    speedup
ultralight      363.919    363.733       1.00x
light           363.916     22.456      16.21x
medium          366.537     24.952      14.69x
full            365.223     23.544      15.51x


## `full` indexes and ordered access

A `full` index stores a global sorted payload. This is the required index tier for direct ordered reuse. `create_csindex()` is just a convenience wrapper for `create_index(kind="full")`.

In [8]:
ordered_dtype = np.dtype([("id", np.int64), ("payload", np.int64)])
ordered_data = np.array(
    [(2, 9), (1, 8), (2, 7), (1, 6), (2, 5), (1, 4), (2, 3), (1, 2)],
    dtype=ordered_dtype,
)
ordered_arr = blosc2.asarray(ordered_data, chunks=(4,), blocks=(2,))
ordered_arr.create_csindex("id")

print("Sorted positions:", ordered_arr.indices(order=["id", "payload"])[:])
print("Sorted rows:")
print(ordered_arr.sort(order=["id", "payload"])[:])

Sorted positions: [7 5 3 1 6 4 2 0]
Sorted rows:
[(1, 2) (1, 4) (1, 6) (1, 8) (2, 3) (2, 5) (2, 7) (2, 9)]


## Expression indexes

You can also index a deterministic scalar expression stream. Expression indexes are matched by normalized expression identity, so the same expression can be reused for filtering and ordered access.

In [9]:
expr_dtype = np.dtype([("x", np.int64), ("payload", np.int32)])
expr_data = np.array([(-8, 0), (5, 1), (-2, 2), (11, 3), (3, 4), (-3, 5), (2, 6), (-5, 7)], dtype=expr_dtype)
expr_arr = blosc2.asarray(expr_data, chunks=(4,), blocks=(2,))
expr_arr.create_expr_index("abs(x)", kind="full", name="abs_x")

ordered_expr = blosc2.lazyexpr("(abs(x) >= 2) & (abs(x) < 8)", expr_arr.fields).where(expr_arr)
print(explain_subset(ordered_expr))
print("Expression-order positions:", ordered_expr.indices(order="abs(x)").compute()[:])

{'will_use_index': True, 'reason': 'multi-field exact indexes selected', 'kind': 'full', 'level': 'exact', 'lookup_path': 'in-memory', 'full_runs': 0}
Expression-order positions: [2 6 4 5 1 7]


## Persistence: automatic or manual?

Index persistence follows the base array by default:

- for a persistent array (`urlpath=...`), `persistent=None` means the index sidecars are persisted automatically,
- for an in-memory array, the index lives only in memory,
- on a persistent array, `persistent=False` keeps the index process-local instead of writing sidecars.

In practice, if you want an index to survive reopen, persist the array and use the default behavior.

In [10]:
persistent_arr = blosc2.asarray(data, urlpath=paths[0], mode="w", chunks=chunks, blocks=blocks)
persistent_descriptor = persistent_arr.create_index(field="id", kind="medium")
show_index_summary("persistent medium", persistent_descriptor)

reopened = blosc2.open(paths[0], mode="a")
print(f"Reopened index count: {len(reopened.indexes)}")
print(f"Persisted sidecar path: {reopened.indexes[0]['reduced']['values_path']}")

persistent medium: kind=medium, persistent=True, ooc=True, stale=False
Reopened index count: 1
Persisted sidecar path: indexing_tutorial_medium.__index__.id.medium.reduced.values.b2nd


## When to rebuild an index

Appending is special-cased and keeps compatible indexes current. General mutation and resize operations do not. After unsupported mutations, the index is marked stale and should be refreshed explicitly with `rebuild_index()`.

In [11]:
mutable_arr = blosc2.asarray(np.arange(20, dtype=np.int64), chunks=(10,), blocks=(5,))
mutable_arr.create_index(kind="full")
mutable_arr[:3] = -1

print("Stale after direct mutation:", mutable_arr.indexes[0]["stale"])
mutable_arr.rebuild_index()
print("Stale after rebuild:", mutable_arr.indexes[0]["stale"])

Stale after direct mutation: True
Stale after rebuild: False


## Recommended workflow for append-heavy `full` indexes

Appending to a `full` index is intentionally cheap: appended tails become sorted runs instead of forcing an immediate rewrite of the compact base sidecars.

That means the recommended workflow is:

1. create a persistent `full` index once,
2. append freely during ingestion,
3. let queries keep working while runs accumulate,
4. call `compact_index()` after ingestion windows or before latency-sensitive read phases.

The next example uses a larger append-heavy array and times the same selective query before and after compaction. The exact query path reports whether it is using a compact lookup layout or a run-aware fallback. After compaction, `full["runs"]` becomes empty again.

In [12]:
append_dtype = np.dtype([("id", np.int64), ("payload", np.int32)])
base_rows = 200_000
append_batch = 500
num_runs = 40

append_data = np.zeros(base_rows, dtype=append_dtype)
append_data["id"] = np.arange(base_rows, dtype=np.int64)
append_data["payload"] = np.arange(base_rows, dtype=np.int32)

append_arr = blosc2.asarray(append_data, urlpath=paths[1], mode="w", chunks=(20_000,), blocks=(4_000,))
append_arr.create_index(field="id", kind="full")

for run in range(num_runs):
    start = 300_000 + run * append_batch
    batch = np.zeros(append_batch, dtype=append_dtype)
    batch["id"] = np.arange(start, start + append_batch, dtype=np.int64)
    batch["payload"] = np.arange(append_batch, dtype=np.int32)
    append_arr.append(batch)

append_query = "(id >= 310_000) & (id < 310_020)"
append_expr = blosc2.lazyexpr(append_query, append_arr.fields).where(append_arr)
before_info = explain_subset(append_expr)
before_ms = median_ms(lambda: append_expr.compute()[:], repeats=5)
print("Before compaction:", before_info)
print("Pending runs:", len(append_arr.indexes[0]["full"]["runs"]))
print(f"Median query time before compaction: {before_ms:.3f} ms")

append_arr.compact_index("id")
append_expr = blosc2.lazyexpr(append_query, append_arr.fields).where(append_arr)
after_info = explain_subset(append_expr)
after_ms = median_ms(lambda: append_expr.compute()[:], repeats=5)
print("After compaction:", after_info)
print("Pending runs:", len(append_arr.indexes[0]["full"]["runs"]))
print(f"Median query time after compaction: {after_ms:.3f} ms")
print(f"Speedup after compaction: {before_ms / after_ms:.2f}x")

Before compaction: {'will_use_index': True, 'reason': 'multi-field exact indexes selected', 'kind': 'full', 'level': 'exact', 'lookup_path': 'run-bounded-ooc', 'full_runs': 40}
Pending runs: 40
Median query time before compaction: 3.250 ms
After compaction: {'will_use_index': True, 'reason': 'multi-field exact indexes selected', 'kind': 'full', 'level': 'exact', 'lookup_path': 'compact-selective-ooc', 'full_runs': 0}
Pending runs: 0
Median query time after compaction: 0.939 ms
Speedup after compaction: 3.46x


## Practical guidance

- Use `medium` when your main goal is faster selective filtering.
- Use `full` when you also want ordered reuse through `sort(order=...)`, `indices(order=...)`, or `itersorted(...)`.
- Persist the base array if you want indexes to survive reopen automatically.
- After unsupported mutations, use `rebuild_index()`.
- For append-heavy `full` indexes, compact explicitly at convenient maintenance boundaries instead of on every append.
- Measure your own workload: compact indexes, predicate selectivity, and ordered access needs all affect which kind is best.


In [13]:
for path in paths:
    blosc2.remove_urlpath(path)